**Organizar Información Adicional de minerales**

En este cuaderno estructuro la información de los siguientes datos para manejarlos en STATA:
- Minería ilegal calculada por SR2021
- Producción minera asociada a regalías

# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

# load GIS setup
from utils.setup_gis_python import *

# Load utils for SR 2021
from utils.utils_for_SR2021 import *

Setup general cargado
Setup GIS cargado


# Definir datos de salida
El archivo de salida se estructura para conservar la misma estructura entre todas las bases de datos

In [2]:
ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [3]:
# Definir DF con la estructura acordada para el proyecto
# La variable global ORDEN_DF ya tiene la lista de todas las columnas ordenadas
# Inicializar el DF vacío
df_salida = pd.DataFrame(
    columns=[ORDEN_DF]
)

# Datos de minería ilegal de SR2021

In [4]:
# Cargar datos
df_SR2021_minerales = pd.read_parquet(
    DATA/'intermediate/e2011_SR2021_mineriaIlegal_armonizado.parquet'   
)

In [5]:
# Explorar datos
print(MSC_SEPARADOR, 'Head')
display(df_SR2021_minerales.head())

print(MSC_SEPARADOR, 'Info')
display(df_SR2021_minerales.info())


-------------------------------- Head


,codmpio,ano,mineral,propmined_illegal_mi,w_analizedpctgarea_mi,analizedareasqkm_mi,areaminedsqkm_mi,propadjminedMi_illegal,areaminedSPsqkm_mi,areaillegalminedSPsqkm_mi,propminedSP_illegal_mi,propminedMi_illegal_prob,newpropminedMi_illegal,c12_propmined_illegal_mi,titulosareaMi_ha,frtitulosareaMi,afterxgold,ind_after,afterxplatinum,precio,regalia_mine,afterxregalia,areamuni_sqkm,propmuni_illegal_mi,mineral_decodificado,texto_busqueda,categoria_armonizada
0,5001,"2,004.00",6,NaN,NaN,0.00,0.00,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,NaN,5.00,0.00,371.53,NaN,iron,6 iron,Metales base
1,5001,"2,010.00",2,100.00,80.00,2.39,0.00,NaN,0.00,0.00,100.00,100.00,NaN,100.00,0.00,0.00,0.00,0.00,0.00,1.35,5.00,0.00,371.53,0.00,platinum,2 platinum,Otros metales preciosos
2,5001,"2,012.00",8,NaN,NaN,0.00,0.00,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,1.00,0.00,NaN,12.00,12.00,371.53,NaN,nickel,8 nickel,Niquel
3,5001,"2,008.00",10,NaN,NaN,0.00,0.00,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,NaN,5.00,0.00,371.53,NaN,potassium,10 potassium,Fertilizantes
4,5001,"2,011.00",8,NaN,NaN,0.00,0.00,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,1.00,0.00,NaN,12.00,12.00,371.53,NaN,nickel,8 nickel,Niquel



-------------------------------- Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132132 entries, 0 to 132131
Data columns (total 27 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   codmpio                    132132 non-null  int32  
 1   ano                        132132 non-null  float32
 2   mineral                    132132 non-null  int8   
 3   propmined_illegal_mi       25946 non-null   float32
 4   w_analizedpctgarea_mi      41950 non-null   float32
 5   analizedareasqkm_mi        131549 non-null  float32
 6   areaminedsqkm_mi           131549 non-null  float32
 7   propadjminedMi_illegal     8817 non-null    float32
 8   areaminedSPsqkm_mi         131571 non-null  float64
 9   areaillegalminedSPsqkm_mi  131571 non-null  float64
 10  propminedSP_illegal_mi     25655 non-null   float32
 11  propminedMi_illegal_prob   15161 non-null   float32
 12  newpropminedMi_illegal     11413 non-null   flo

None

In [6]:
# Los principales retos de esta base son (1) identificar las columnas relevantes para mi investigación
# y (2) entender el significado exacto de las variables seleccionadas


## Dar estructura al DF

In [7]:
# Explorar como se reparten los minerales idetificados por SR2021 en
# las categorías armonizadas
df_SR2021_minerales.groupby(['categoria_armonizada', 'mineral_decodificado'])['codmpio'].count()

categoria_armonizada     mineral_decodificado
Carbon                   coal                    12012
Cobre                    copper                  12012
Fertilizantes            phosphate               12012
                         potassium               12012
Metales base             columbite               12012
                         iron                    12012
Minerales industriales   magnesium               12012
Niquel                   nickel                  12012
Oro                      gold                    12012
Otros                    uranium                 12012
Otros metales preciosos  platinum                12012
Name: codmpio, dtype: int64

In [8]:
# hacer copia de los datos
panel_SR2021_minerales = df_SR2021_minerales.copy()


# Garantizar el tipo de cada variable
panel_SR2021_minerales['codmpio'] = (
    panel_SR2021_minerales['codmpio']
    .astype("Int64")
    .astype("string").
    str.zfill(5))

panel_SR2021_minerales['ano'] = (
    panel_SR2021_minerales['ano']
    .astype("Int64"))


# Identificar variables relevantes
variables_de_interes = ['areaillegalminedSPsqkm_mi', 'newpropminedMi_illegal', 'propmuni_illegal_mi']

# Como cada observacion de (municipio, año, categoría mineral) puede tener varias filas
# por ejemplo, en metales base hay registros de "Columbite" y "iron".
# Los resultados de la celda de arriba muestran que solo afectaría a lasc ategorías de "Metales base" y "Fertilizantes"
panel_SR2021_minerales = panel_SR2021_minerales.groupby(['codmpio', 'ano', 'categoria_armonizada'])[variables_de_interes].sum()
panel_SR2021_minerales = panel_SR2021_minerales.reset_index()

# Dar estructura acordada para el panel
panel_SR2021_minerales = panel_SR2021_minerales.melt(
    id_vars=['codmpio', 'ano', 'categoria_armonizada'],
    value_vars=variables_de_interes,
    var_name=COL_VARIABLE_MEDICION,
    value_name=COL_VALOR
)

# Asignar nombres estandar del panel
panel_SR2021_minerales = panel_SR2021_minerales.rename(
    columns={
        'codmpio':COL_ID_MUNICIPIO,
        'ano':COL_ANNO,
        'categoria_armonizada':COL_VARIABLE_SUJETO,
    }
)

# Completar información del panel
panel_SR2021_minerales[COL_CLASIFICACION_ECONOMETRIA]='X: Mineria ilegal'
panel_SR2021_minerales[COL_VARIABLE_DETALLE]='calculos_SR_2021'
panel_SR2021_minerales[COL_VARIABLE_DESCRIPCION]= ('Mineria ilegal.' + 
    ' Mineral: ' + panel_SR2021_minerales[COL_VARIABLE_SUJETO] +
    ' Medición: ' + panel_SR2021_minerales[COL_VARIABLE_MEDICION] +
    ' Detalle: ' + panel_SR2021_minerales[COL_VARIABLE_DETALLE]
)

ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [9]:
# Asignar nombres a las variables
nombre_variable_concepto = 'mineIleg'

# Abreviaciones (máximo 6 caracteres) para los nombres de los minerales
abreviaciones_minerales = {
    "Oro": "oro",
    "Carbon": "carbon",
    "Niquel": "niquel",
    "Cobre": "cobre",
    "Otros metales preciosos": "oPreci",
    "Gemas": "gemas",
    "Otros": "otros",
    "Metales base": "metBas",
    "Materiales construccion": "mConst",
    "Minerales industriales": "mIndus",
    "Fertilizantes": "fertlz"
}
nombre_variable_sujeto = panel_SR2021_minerales[COL_VARIABLE_SUJETO].map(abreviaciones_minerales)


In [10]:
display(panel_SR2021_minerales[COL_VARIABLE_MEDICION].unique())
# Definir diccionarios para generar nombres de variables
abreviaciones_medicion = {
    'areaillegalminedSPsqkm_mi': 'area',
    'newpropminedMi_illegal': 'nwPrp',
    'propmuni_illegal_mi': 'prpMun'
}
nombre_variable_medicion = panel_SR2021_minerales[COL_VARIABLE_MEDICION].map(abreviaciones_medicion)

abreviaciones_unidades = {
    'areaillegalminedSPsqkm_mi': 'km2',
    'newpropminedMi_illegal': 'pct',
    'propmuni_illegal_mi': 'pct'
}
nombre_variable_unidades = panel_SR2021_minerales[COL_VARIABLE_MEDICION].map(abreviaciones_unidades)

array(['areaillegalminedSPsqkm_mi', 'newpropminedMi_illegal',
       'propmuni_illegal_mi'], dtype=object)

In [11]:
display(panel_SR2021_minerales[COL_VARIABLE_DETALLE].unique())
nombre_variable_detalle = 'SR21'

array(['calculos_SR_2021'], dtype=object)

In [12]:
# Asignar nombre de variable con la estructura definida
panel_SR2021_minerales[COL_NOMBRE_DE_VARIABLE] = (nombre_variable_concepto + '_' + 
 nombre_variable_sujeto + '_' + 
 nombre_variable_medicion + '_' +
 nombre_variable_detalle + '_' +
 nombre_variable_unidades)

In [13]:
# Verificar que la longitud del string del nombre de la variable no es superior a 32 caracteres
panel_SR2021_minerales[COL_NOMBRE_DE_VARIABLE].str.len().max()

31

In [14]:
panel_SR2021_minerales[ORDEN_DF].head(3)

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2004,mineIleg_carbon_area_SR21_km2,Carbon,areaillegalminedSPsqkm_mi,calculos_SR_2021,Mineria ilegal. Mineral: Carbon Medición: area...,0.00,X: Mineria ilegal
1,05001,2004,mineIleg_cobre_area_SR21_km2,Cobre,areaillegalminedSPsqkm_mi,calculos_SR_2021,Mineria ilegal. Mineral: Cobre Medición: areai...,0.00,X: Mineria ilegal
2,05001,2004,mineIleg_fertlz_area_SR21_km2,Fertilizantes,areaillegalminedSPsqkm_mi,calculos_SR_2021,Mineria ilegal. Mineral: Fertilizantes Medició...,0.00,X: Mineria ilegal


# Datos de producción minera legal asociada a regalías

In [15]:
# Cargar datos
df_UPME_produccionRegalias = pd.read_parquet(
    DATA/'intermediate/e2011_UPME_produccionRegalias_armonizado.parquet'
)

print(MSC_SEPARADOR, "Datos originales")
# Exploración inicial
df_UPME_produccionRegalias.head()


-------------------------------- Datos originales


,Año,Mes,Mineral,UnidadMedida,Regalias ($),Produccion,Codigo Municipio,Municipio,Codigo Departamento,Departamento,texto_busqueda,categoria_armonizada
0,2026,1,ORO,GRAMOS,"1,013,843,150.81","634,108.02",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
1,2025,12,ORO,GRAMOS,"969,828,842.90","690,459.19",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
2,2025,11,ORO,GRAMOS,"746,090,262.90","612,155.49",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
3,2025,10,ORO,GRAMOS,"705,220,681.15","567,809.15",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
4,2025,9,ORO,GRAMOS,"885,010,981.44","788,920.45",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro


## Procesamiento

In [16]:
# Conservar solo las columnas de interés
columnas_para_conservar = ['categoria_armonizada', 'Año', 'Codigo Municipio', 'Mineral', 'UnidadMedida', 'Regalias ($)', 'Produccion']
df_UPME_produccionRegalias = df_UPME_produccionRegalias[columnas_para_conservar]


# Asegurarse que la columna del codigo de municipio sea un string de 5 dígitos
df_UPME_produccionRegalias["Codigo Municipio"] = (
    df_UPME_produccionRegalias["Codigo Municipio"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)


# Agregar valores mensuales en valores anuales
df_UPME_produccionRegalias_anual = df_UPME_produccionRegalias.groupby(
    ['categoria_armonizada', 'Año', 'Codigo Municipio', 'Mineral', 'UnidadMedida'])[['Regalias ($)', 'Produccion']].sum().reset_index()


print(MSC_SEPARADOR, "Datos procesados")
display(df_UPME_produccionRegalias_anual.head())


-------------------------------- Datos procesados


,categoria_armonizada,Año,Codigo Municipio,Mineral,UnidadMedida,Regalias ($),Produccion
0,Carbon,2012,05030,CARBON,TONELADAS,"1,051,738,901.34","237,922.24"
1,Carbon,2012,05036,CARBON,TONELADAS,"54,852,809.23","6,826.35"
2,Carbon,2012,05282,CARBON,TONELADAS,"294,127,636.19","73,610.71"
3,Carbon,2012,05809,CARBON,TONELADAS,"695,137,060.68","181,140.02"
4,Carbon,2012,05861,CARBON,TONELADAS,"161,453,262.02","6,353.69"


In [17]:
# Mostrar los valores en los que se mide la producción de diferentes materiales
# Estoy explorando sie puedo sumar la producción de todos los mienerales dentro de la misma categoría armonizada
# Hay algunos en los que es posible y otros y en los que se podrían convertir unidades de volumen a unidades de masa
# por ahora, para la producción solo voy a agregar los que tengan la misma unidadd de medida
# En refinamientos posteriores podría hacer la conversión de unidades de la que hablo arriba.
with pd.option_context("display.max_rows", None):
    display(
        df_UPME_produccionRegalias_anual
        .groupby(["categoria_armonizada", "UnidadMedida"])["Mineral"]
        .value_counts()
    )

categoria_armonizada     UnidadMedida    Mineral                                                        
Carbon                   TONELADAS       CARBON                                                             1093
                                         CARBON TERMICO                                                      623
                                         CARBON METALURGICO                                                  479
                                         CARBON ANTRACITA                                                     13
Cobre                    KILOGRAMOS      COBRE                                                                58
Fertilizantes            TONELADAS       ROCA FOSFORICA                                                      119
Gemas                    QUILATES        ESMERALDAS EN BRUTO                                                  94
                                         ESMERALDAS TALLADAS                                            

## Valor de regalías

In [18]:
# Agregar valor de las regalías
df_UPME_produccionRegalias_valorRegalias = df_UPME_produccionRegalias_anual.groupby(
    ["categoria_armonizada", 'Año', 'Codigo Municipio'])['Regalias ($)'].sum().reset_index()

### Dar estructura al DF

In [19]:
# hacer copia de los datos
panel_valorRegalias = df_UPME_produccionRegalias_valorRegalias.copy()

# Asignar nombres acordados a las columnas
panel_valorRegalias = panel_valorRegalias.rename(
    columns={
        'categoria_armonizada':COL_VARIABLE_SUJETO,
        'Año':COL_ANNO,
        'Codigo Municipio': COL_ID_MUNICIPIO,
        'Regalias ($)': COL_VALOR
    }
)

# Completar información de estructura del panel
panel_valorRegalias[COL_CLASIFICACION_ECONOMETRIA] = 'X: Mineria legal'
panel_valorRegalias[COL_NOMBRE_DE_VARIABLE] = ''
panel_valorRegalias[COL_VARIABLE_DETALLE] = 'Valor asociado a regalías en COP Corrientes'
panel_valorRegalias[COL_VARIABLE_MEDICION] = 'produccionRegalias_copCorrientes' # ¿Habría que pasarlo a plata constante?
panel_valorRegalias[COL_VARIABLE_DESCRIPCION] = ('Producción legal.' +
                                                ' Mineral: ' + panel_valorRegalias[COL_VARIABLE_SUJETO] +
                                                 ' Medicion: ' + panel_valorRegalias[COL_VARIABLE_MEDICION] +
                                                 ' Detalle: '+ panel_valorRegalias[COL_VARIABLE_DETALLE]
                                                )

display(panel_valorRegalias[ORDEN_DF].head(3))

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"1,051,738,901.34",X: Mineria legal
1,05036,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"54,852,809.23",X: Mineria legal
2,05282,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"294,127,636.19",X: Mineria legal


In [20]:
# Asignar nombres a las variables
nombre_variable_concepto = 'mineLegl'
nombre_variable_sujeto = panel_valorRegalias[COL_VARIABLE_SUJETO].map(abreviaciones_minerales)


In [21]:
display(panel_valorRegalias[COL_VARIABLE_MEDICION].unique())
# Definir diccionarios para generar nombres de variables

abreviaciones_medicion = {
    'produccionRegalias_copCorrientes': 'pRegls',
}
nombre_variable_medicion = panel_valorRegalias[COL_VARIABLE_MEDICION].map(abreviaciones_medicion)

abreviaciones_unidades = {
    'produccionRegalias_copCorrientes': 'COP', # COP corrientes
}
nombre_variable_unidades = panel_valorRegalias[COL_VARIABLE_MEDICION].map(abreviaciones_unidades)

array(['produccionRegalias_copCorrientes'], dtype=object)

In [22]:
display(panel_valorRegalias[COL_VARIABLE_DETALLE].unique())
nombre_variable_detalle = 'valr'

array(['Valor asociado a regalías en COP Corrientes'], dtype=object)

In [23]:
# Asignar nombre de variable con la estructura definida
panel_valorRegalias[COL_NOMBRE_DE_VARIABLE] = (nombre_variable_concepto + '_' + 
 nombre_variable_sujeto + '_' + 
 nombre_variable_medicion + '_' +
 nombre_variable_detalle + '_' +
 nombre_variable_unidades)

In [24]:
# Verificar que la longitud del string del nombre de la variable no es superior a 32 caracteres
panel_valorRegalias[COL_NOMBRE_DE_VARIABLE].str.len().max()

31

In [25]:
panel_valorRegalias[ORDEN_DF].head(3)

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"1,051,738,901.34",X: Mineria legal
1,05036,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"54,852,809.23",X: Mineria legal
2,05282,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"294,127,636.19",X: Mineria legal


## Producción

In [26]:
# Agregar producción
# En cada "categoria_armonizada", voy a elegir el mayor conteo de registros de producción.
# Elijo esa unidad de medida y agrego en esa unidad de medida

# Econtrar el conteo de observaciones
unidades_de_medida_conteo = df_UPME_produccionRegalias_anual.groupby(["categoria_armonizada"])["UnidadMedida"].value_counts().reset_index()
display(unidades_de_medida_conteo)

# Conservar solo la unidad de medida con más registros en cada categoria armonziada
unidades_principales = (
    unidades_de_medida_conteo.loc[
        unidades_de_medida_conteo
        .groupby("categoria_armonizada")["count"]
        .idxmax()
    ]
    .reset_index(drop=True)
)
print(MSC_SEPARADOR, "Unidades principales")
display(unidades_principales)

,categoria_armonizada,UnidadMedida,count
0,Carbon,TONELADAS,2208
1,Cobre,KILOGRAMOS,58
2,Fertilizantes,TONELADAS,119
3,Gemas,QUILATES,254
4,Materiales construccion,METROS CUBICOS,11391
5,Materiales construccion,TONELADAS,1009
6,Materiales construccion,KILOGRAMOS,9
7,Metales base,TONELADAS,118
8,Metales base,KILOGRAMOS,71
9,Minerales industriales,TONELADAS,3013



-------------------------------- Unidades principales


,categoria_armonizada,UnidadMedida,count
0,Carbon,TONELADAS,2208
1,Cobre,KILOGRAMOS,58
2,Fertilizantes,TONELADAS,119
3,Gemas,QUILATES,254
4,Materiales construccion,METROS CUBICOS,11391
5,Metales base,TONELADAS,118
6,Minerales industriales,TONELADAS,3013
7,Niquel,LIBRAS,42
8,Oro,GRAMOS,2095
9,Otros,METROS CUBICOS,51


In [27]:
# Para la producción voy a agregar únicamente las categorías armonizadas con la misma uniadd de medida
# Desecho las demás observaciones

df_UPME_produccionRegalias_anual_mismaUnidad = (
    df_UPME_produccionRegalias_anual
    .merge(
        unidades_principales[
            ["categoria_armonizada", "UnidadMedida"]
        ],
        on=["categoria_armonizada", "UnidadMedida"],
        how="inner",
        validate="many_to_one"
    )
)

In [28]:
# Agregar Producción asociada a regalías

df_UPME_produccionRegalias_produccionAsociadaRegalias = df_UPME_produccionRegalias_anual_mismaUnidad.groupby(
    ["categoria_armonizada", 'Año', 'Codigo Municipio'])[['Produccion', 'UnidadMedida']].aggregate({'Produccion':'sum', 'UnidadMedida':'first'}).reset_index()
display(df_UPME_produccionRegalias_produccionAsociadaRegalias)

,categoria_armonizada,Año,Codigo Municipio,Produccion,UnidadMedida
0,Carbon,2012,05030,"237,922.24",TONELADAS
1,Carbon,2012,05036,"6,826.35",TONELADAS
2,Carbon,2012,05282,"73,610.71",TONELADAS
3,Carbon,2012,05809,"181,140.02",TONELADAS
4,Carbon,2012,05861,"6,353.69",TONELADAS
...,...,...,...,...,...
13087,Otros metales preciosos,2026,73067,681.85,GRAMOS
13088,Otros metales preciosos,2026,73168,0.00,GRAMOS
13089,Otros metales preciosos,2026,73270,0.00,GRAMOS
13090,Otros metales preciosos,2026,73411,"173,689.24",GRAMOS


### Dar estructura al DF

In [29]:
# hacer copia de los datos
panel_produccionAsociadaRegalias = df_UPME_produccionRegalias_produccionAsociadaRegalias.copy()

# Asignar nombres acordados a las columnas
panel_produccionAsociadaRegalias = panel_produccionAsociadaRegalias.rename(
    columns={
        'categoria_armonizada':COL_VARIABLE_SUJETO,
        'Año':COL_ANNO,
        'Codigo Municipio': COL_ID_MUNICIPIO,
        'Produccion': COL_VALOR,
        'UnidadMedida': COL_VARIABLE_MEDICION
    }
)

# Completar información de estructura del panel
panel_produccionAsociadaRegalias[COL_CLASIFICACION_ECONOMETRIA] = 'X: Mineria legal'
panel_produccionAsociadaRegalias[COL_NOMBRE_DE_VARIABLE] = ''
panel_produccionAsociadaRegalias[COL_VARIABLE_DETALLE] = 'Producción asociada a regalías en cantidad'
panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION] = 'produccionRegalias_' + panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION]
panel_produccionAsociadaRegalias[COL_VARIABLE_DESCRIPCION] = ('Producción legal.' +
                                                ' Mineral: ' + panel_produccionAsociadaRegalias[COL_VARIABLE_SUJETO] +
                                                 ' Medicion: ' + panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION] +
                                                 ' Detalle: '+ panel_produccionAsociadaRegalias[COL_VARIABLE_DETALLE]
                                                )

display(panel_produccionAsociadaRegalias[ORDEN_DF].head(3))

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"237,922.24",X: Mineria legal
1,05036,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"6,826.35",X: Mineria legal
2,05282,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"73,610.71",X: Mineria legal


In [30]:
# Asignar nombres a las variables
nombre_variable_concepto = 'mineLegl'
nombre_variable_sujeto = panel_produccionAsociadaRegalias[COL_VARIABLE_SUJETO].map(abreviaciones_minerales)


In [31]:
display(panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION].unique())
# Definir diccionarios para generar nombres de variables

nombre_variable_medicion = 'pRegls'

abreviaciones_unidades = {
    'produccionRegalias_TONELADAS': 'ton',
    'produccionRegalias_KILOGRAMOS': 'kg',
    'produccionRegalias_QUILATES': 'qlt',
    'produccionRegalias_METROS CUBICOS': 'm3',
    'produccionRegalias_LIBRAS': 'lb',
    'produccionRegalias_GRAMOS': 'gr'
}
nombre_variable_unidades = panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION].map(abreviaciones_unidades)

array(['produccionRegalias_TONELADAS', 'produccionRegalias_KILOGRAMOS',
       'produccionRegalias_QUILATES', 'produccionRegalias_METROS CUBICOS',
       'produccionRegalias_LIBRAS', 'produccionRegalias_GRAMOS'],
      dtype=object)

In [32]:
display(panel_produccionAsociadaRegalias[COL_VARIABLE_DETALLE].unique())
nombre_variable_detalle = 'prod'

array(['Producción asociada a regalías en cantidad'], dtype=object)

In [33]:
# Asignar nombre de variable con la estructura definida
panel_produccionAsociadaRegalias[COL_NOMBRE_DE_VARIABLE] = (nombre_variable_concepto + '_' + 
 nombre_variable_sujeto + '_' + 
 nombre_variable_medicion + '_' +
 nombre_variable_detalle + '_' +
 nombre_variable_unidades)

In [34]:
# Verificar que la longitud del string del nombre de la variable no es superior a 32 caracteres
panel_produccionAsociadaRegalias[COL_NOMBRE_DE_VARIABLE].str.len().max()

31

In [35]:
panel_produccionAsociadaRegalias[ORDEN_DF].head(3)

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,mineLegl_carbon_pRegls_prod_ton,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"237,922.24",X: Mineria legal
1,05036,2012,mineLegl_carbon_pRegls_prod_ton,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"6,826.35",X: Mineria legal
2,05282,2012,mineLegl_carbon_pRegls_prod_ton,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en cantidad,Producción legal. Mineral: Carbon Medicion: pr...,"73,610.71",X: Mineria legal


# organizar panel de salida

In [36]:
# Bases a exportar
df_salida = pd.concat(
    [
        panel_valorRegalias,
        panel_produccionAsociadaRegalias,
        panel_SR2021_minerales
    ]
)

In [38]:
# Dar forma al Dataframe para leerlo en STATA
df_salida_STATA = df_salida.pivot(
        index=[COL_ANNO, COL_ID_MUNICIPIO],
        columns=COL_NOMBRE_DE_VARIABLE,
        values=COL_VALOR
    ).reset_index().rename_axis(columns=None)

In [39]:
# Mostrar base de datos de la salida
df_salida[ORDEN_DF].head()

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"1,051,738,901.34",X: Mineria legal
1,05036,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"54,852,809.23",X: Mineria legal
2,05282,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"294,127,636.19",X: Mineria legal
3,05809,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"695,137,060.68",X: Mineria legal
4,05861,2012,mineLegl_carbon_pRegls_valr_COP,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"161,453,262.02",X: Mineria legal


In [40]:
# Explorar variables disponibles por mineral
df_salida.groupby([COL_CLASIFICACION_ECONOMETRIA])[COL_VARIABLE_SUJETO].value_counts().reset_index()

,clasificacion_econometria,variable_sujeto,count
0,X: Mineria ilegal,Carbon,36036
1,X: Mineria ilegal,Fertilizantes,36036
2,X: Mineria ilegal,Metales base,36036
3,X: Mineria ilegal,Minerales industriales,36036
4,X: Mineria ilegal,Niquel,36036
5,X: Mineria ilegal,Oro,36036
6,X: Mineria ilegal,Otros,36036
7,X: Mineria ilegal,Otros metales preciosos,36036
8,X: Mineria ilegal,Cobre,36036
9,X: Mineria legal,Materiales construccion,11323


In [41]:
# Explorar variables disponibles por mineral
df_salida.groupby([COL_CLASIFICACION_ECONOMETRIA, COL_VARIABLE_MEDICION])[COL_VARIABLE_SUJETO].value_counts().reset_index()

,clasificacion_econometria,variable_medicion,variable_sujeto,count
0,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Carbon,12012
1,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Cobre,12012
2,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Otros,12012
3,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Fertilizantes,12012
4,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Metales base,12012
5,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Niquel,12012
6,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Oro,12012
7,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Minerales industriales,12012
8,X: Mineria ilegal,areaillegalminedSPsqkm_mi,Otros metales preciosos,12012
9,X: Mineria ilegal,newpropminedMi_illegal,Niquel,12012


In [42]:
# Exportar base de datos en parquet
df_salida.to_parquet(DATA/'intermediate/e2101_panel_InfoAdicional_Minerales.parquet')

In [46]:
df_salida_STATA.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18614 entries, 0 to 18613
Data columns (total 51 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   anno                             18614 non-null  Int64  
 1   codigo_dane_municipio            18614 non-null  string 
 2   mineIleg_carbon_area_SR21_km2    12012 non-null  float64
 3   mineIleg_carbon_nwPrp_SR21_pct   12012 non-null  float64
 4   mineIleg_carbon_prpMun_SR21_pct  12012 non-null  float64
 5   mineIleg_cobre_area_SR21_km2     12012 non-null  float64
 6   mineIleg_cobre_nwPrp_SR21_pct    12012 non-null  float64
 7   mineIleg_cobre_prpMun_SR21_pct   12012 non-null  float64
 8   mineIleg_fertlz_area_SR21_km2    12012 non-null  float64
 9   mineIleg_fertlz_nwPrp_SR21_pct   12012 non-null  float64
 10  mineIleg_fertlz_prpMun_SR21_pct  12012 non-null  float64
 11  mineIleg_mIndus_area_SR21_km2    12012 non-null  float64
 12  mineIleg_mIndus_nw

In [48]:
# Exportar base de datos para stata

# en pandas se esta manejando objetos "string" que STATA no soporta. Por eso es necesario convertirlo a "str"
df_stata = df_salida_STATA.copy()
columnas_string = df_stata.select_dtypes(include="string").columns

df_stata[columnas_string] = (
    df_stata[columnas_string]
    .astype(object)
    .where(df_stata[columnas_string].notna(), None)
)
df_stata.to_stata(DATA/'intermediate/e2101_panel_InfoAdicional_Minerales.dta')

# Verificaciones adicionales
En las siguientes secciones exploro consideraciones sobre los datos trabajados en este cuaderno que van surgiendo a medida que avanzo 

In [2]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

# load GIS setup
from utils.setup_gis_python import *

# Load utils for SR 2021
from utils.utils_for_SR2021 import *

## Producción regalias UPME - Verificaciones

## Explorar datos de produccion en 0
En el análisis de STATA e8100_explorar_primera_etapa encontré necesario explorar si los ceros de producción de oro se pueden interpretar como producción = 0 o como datos faltantes. Acá exploro ese problema

In [18]:
# Cargar datos
df_stata = pd.read_stata(
    DATA/'intermediate/e2101_panel_InfoAdicional_Minerales.dta'
)

In [19]:
# Identificar los municipios con registros de produccion = 0
datos_cero_produccion = df_stata[df_stata['mineLegl_oro_pRegls_prod_gr']==0]

In [20]:
# Explorar los datos en cuestión
display(
    datos_cero_produccion[[
        COL_ID_MUNICIPIO,
        'mineLegl_oro_pRegls_prod_gr',
        'mineLegl_oro_pRegls_valr_COP']]
)
# Los datos de producción cero tienen un valor COP mayor que 0
# Si no hubiera producción, no debería existir un valor COP de regalías
# Se interpreta que los valores de producción 0 son errores de registro

,codigo_dane_municipio,mineLegl_oro_pRegls_prod_gr,mineLegl_oro_pRegls_valr_COP
14329,41001,0.00,"3,038.84"
15854,13030,0.00,"813,630.65"
15981,19022,0.00,"4,471,977.00"
16230,68001,0.00,838.20
16249,68307,0.00,161.80
16853,68406,0.00,"5,112,459.70"
17553,05212,0.00,"263,705.92"
17705,17380,0.00,"44,678.71"
17709,17513,0.00,"2,789,609.14"
17883,52427,0.00,"60,355,323.46"


In [27]:
# Extraer datos
id_municipios_cero_produccion = datos_cero_produccion[COL_ID_MUNICIPIO].unique()
datos_municipios_cero_prod = df_stata[df_stata[COL_ID_MUNICIPIO].isin(id_municipios_cero_produccion)]

datos_municipios_cero_prod.head(2)

,index,anno,codigo_dane_municipio,mineIleg_carbon_area_SR21_km2,mineIleg_carbon_nwPrp_SR21_pct,mineIleg_carbon_prpMun_SR21_pct,mineIleg_cobre_area_SR21_km2,mineIleg_cobre_nwPrp_SR21_pct,mineIleg_cobre_prpMun_SR21_pct,mineIleg_fertlz_area_SR21_km2,mineIleg_fertlz_nwPrp_SR21_pct,mineIleg_fertlz_prpMun_SR21_pct,mineIleg_mIndus_area_SR21_km2,mineIleg_mIndus_nwPrp_SR21_pct,mineIleg_mIndus_prpMun_SR21_pct,mineIleg_metBas_area_SR21_km2,mineIleg_metBas_nwPrp_SR21_pct,mineIleg_metBas_prpMun_SR21_pct,mineIleg_niquel_area_SR21_km2,mineIleg_niquel_nwPrp_SR21_pct,mineIleg_niquel_prpMun_SR21_pct,mineIleg_oPreci_area_SR21_km2,mineIleg_oPreci_nwPrp_SR21_pct,mineIleg_oPreci_prpMun_SR21_pct,mineIleg_oro_area_SR21_km2,mineIleg_oro_nwPrp_SR21_pct,mineIleg_oro_prpMun_SR21_pct,mineIleg_otros_area_SR21_km2,mineIleg_otros_nwPrp_SR21_pct,mineIleg_otros_prpMun_SR21_pct,mineLegl_carbon_pRegls_prod_ton,mineLegl_carbon_pRegls_valr_COP,mineLegl_cobre_pRegls_prod_kg,mineLegl_cobre_pRegls_valr_COP,mineLegl_fertlz_pRegls_prod_ton,mineLegl_fertlz_pRegls_valr_COP,mineLegl_gemas_pRegls_prod_qlt,mineLegl_gemas_pRegls_valr_COP,mineLegl_mConst_pRegls_prod_m3,mineLegl_mConst_pRegls_valr_COP,mineLegl_mIndus_pRegls_prod_ton,mineLegl_mIndus_pRegls_valr_COP,mineLegl_metBas_pRegls_prod_ton,mineLegl_metBas_pRegls_valr_COP,mineLegl_niquel_pRegls_prod_lb,mineLegl_niquel_pRegls_valr_COP,mineLegl_oPreci_pRegls_prod_gr,mineLegl_oPreci_pRegls_valr_COP,mineLegl_oro_pRegls_prod_gr,mineLegl_oro_pRegls_valr_COP,mineLegl_otros_pRegls_prod_m3,mineLegl_otros_pRegls_valr_COP
16,16,2004,05079,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.00,0.01,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,36,2004,05190,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# Explorar la serie completa de los municipios en cuestión
print(MSC_SEPARADOR, 'Producción gramos')
display(
    municipios_cero_produccion.pivot(
    index=COL_ANNO,
    columns=COL_ID_MUNICIPIO,
    values='mineLegl_oro_pRegls_prod_gr'
    )
)

print(MSC_SEPARADOR, 'Regalias COP')
display(
    municipios_cero_produccion.pivot(
    index=COL_ANNO,
    columns=COL_ID_MUNICIPIO,
    values='mineLegl_oro_pRegls_valr_COP'
    )
)


-------------------------------- Producción gramos


codigo_dane_municipio,05079,05190,05212,13030,13600,13683,17380,17513,17777,19022,41001,52427,68001,68307,68406,68615,73168,73270,73870
anno,,,,,,,,,,,,,,,,,,,
2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN
2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN
2024,NaN,NaN,0.00,NaN,NaN,NaN,0.00,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00
2025,NaN,NaN,NaN,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00
2026,0.00,0.00,0.00,NaN,0.00,NaN,NaN,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00



-------------------------------- Regalias COP


codigo_dane_municipio,05079,05190,05212,13030,13600,13683,17380,17513,17777,19022,41001,52427,68001,68307,68406,68615,73168,73270,73870
anno,,,,,,,,,,,,,,,,,,,
2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"3,038.84",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021,NaN,NaN,NaN,"813,630.65",NaN,NaN,NaN,NaN,NaN,"4,471,977.00",NaN,NaN,838.20,161.80,NaN,NaN,NaN,NaN,NaN
2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"5,112,459.70",NaN,NaN,NaN,NaN
2024,NaN,NaN,"263,705.92",NaN,NaN,NaN,"44,678.71","2,789,609.14",NaN,NaN,NaN,"60,355,323.46",NaN,NaN,NaN,NaN,NaN,NaN,"54,476,165.87"
2025,NaN,NaN,NaN,NaN,"257,126,072.54","6,033,571.17",NaN,"10,004,001.80",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"11,791,822.59","134,300,997.95"
2026,"11,806,983.73","148,753,396.66","6,436,347.11",NaN,"1,077,940.29",NaN,NaN,"1,696,795.02","2,232,625.02",NaN,NaN,NaN,NaN,NaN,NaN,"12,207,250.16","1,912,249.64","2,112,800.06","24,063,383.97"


Como el mismo (municipio, año) registra (1) regalías y producción de 0 gramos, se entiende que el 0 es un error de registro.

Decido elminar los ceros en STATA